# Control plots for measurements on BGS mocks

> Note : We are not using the Observable class here, as we want to visualize the measurements as they are done - even incomplet, and the compression methods will crash if all the data is not present.

## 0- Setup

In [ ]:
import lsstypes
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

from acm.utils.default import cosmo_list
from acm.utils.abacus import get_abacus_phases
from acm.utils.plotting import plot_parameters_histogram, plot_parameters_triangle

from visual_tools import *

In [ ]:
Mr = -21.35
target = 1.2e-3 # Mpc^-3 h^3
parameters_dir = Path(f'/pscratch/sd/s/sbouchar/acm/bgs{Mr}/parameters/')
measurements_dir = Path(f'/pscratch/sd/s/sbouchar/acm/bgs{Mr}/measurements/')

# SecondGen measurements directory
ref_dir = Path(f'/pscratch/sd/s/sbouchar/SecondGen/CubicBox/BGS/z0.200/AbacusSummit_base_c000_ph000/measurements/Mr{Mr}') 

## 1- Parameter sampling

### 1.1- Cosmological parameters
See [AbacusSummit](https://abacussummit.readthedocs.io/en/latest/cosmologies.html) cosmologies for details.

### 1.2- HOD parameters

#### Compare sampling for BGS and EMC

In [ ]:
hod_params_bgs = load_hod_params(str(parameters_dir / 'hod_params'))
hod_params_bgs = np.concatenate([hod_params_bgs[key] for key in sorted(hod_params_bgs.keys())])

hod_params_emc = load_hod_params('/pscratch/sd/e/epaillas/emc/hod_params/yuan23/cosmo_split/')
hod_params_emc = np.concatenate([hod_params_emc[key] for key in sorted(hod_params_emc.keys())])

fig, ax = plot_parameters_histogram(
    parameters = [hod_params_bgs, hod_params_emc], 
    names = hod_params_bgs.dtype.names, 
    labels = ['BGS samples', 'EMC samples'], 
    bins = 50, 
    alpha = 0.7, 
    # density = True,
)
ax[0].legend()
fig.tight_layout()

In [ ]:
hod_params_bgs = load_hod_params(str(parameters_dir / 'hod_params'))
hod_params_bgs = np.concatenate([hod_params_bgs[key] for key in sorted(hod_params_bgs.keys())])

params = ['logM_cut', 'logM_1', 'sigma', 'alpha', 'kappa']
fig, ax = plot_parameters_histogram(
    parameters = [hod_params_bgs,], 
    names = params, 
    labels = ['BGS samples'], 
    bins = 50, 
    alpha = 0.7, 
    density = True,
)

HOD_params = dict(
    Mr18 = dict(
        logM_cut = 11.16745, 
        logM_1 = 12.92566, 
        kappa = 10**(5.08046 - 11.16745),  # kappa = M0/Mcut = 10^(logM0 - logMcut)
        sigma = np.log10(0.02969), 
        alpha = 1.40723
    ),
    Mr19 = dict(
        logM_cut = 11.42954, 
        logM_1 = 13.10684, 
        kappa = 10**(7.16418 - 11.42954), # kappa = M0/Mcut = 10^(logM0 - logMcut)
        sigma = np.log10(0.04897), 
        alpha = 1.40762,
    ),
    Mr20 = dict(
        logM_cut = 11.84549, 
        logM_1 = 13.40267, 
        kappa = 10**(9.24791-11.84549), # kappa = M0/Mcut = 10^(logM0 - logMcut) 
        sigma = np.log10(0.10304), # Passing logsigma to BoxHOD 
        alpha = 1.41008
    )
)

ls = ['-', '--', ':']
for i, Mr in enumerate([-18, -19, -20]):
    for j, key in enumerate(params):
        ax[j].axvline(HOD_params[f'Mr{abs(Mr)}'][key], color='k', linestyle=ls[i])

handles = []
for i, Mr in enumerate([-18, -19, -20]):
    handle = plt.Line2D([], [], color='k', linestyle=ls[i], label=f'Mr<{Mr} HOD')
    handles.append(handle)

ax[0].legend(handles=handles)
fig.tight_layout()

#### Compare cosmologies for BGS

In [ ]:
hod_params_cosmo = load_hod_params(str(parameters_dir / 'hod_params'))
hod_params_cosmo = [hod_params_cosmo[f'Bouchard25_c{c:03d}'] for c in sorted(cosmo_list)]

fig, ax = plot_parameters_histogram(
    parameters = hod_params_cosmo, 
    names = hod_params_cosmo[0].dtype.names, 
    labels = [f'c{c:03d}' for c in sorted(cosmo_list)],
    bins = 50, 
    alpha = 0.3, 
    # density = True,
)
# ax[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small', title='Cosmology')
fig.suptitle('HOD Parameter Distributions Across Cosmologies', fontsize=10, y=1.00)
fig.tight_layout()

#### Check Cosmological parameters

In [ ]:
cosmo_params = load_hod_params(str(parameters_dir / 'cosmo+hod_params'))
cosmo_params = np.concatenate([cosmo_params[key] for key in sorted(cosmo_params.keys())])

fig, ax = plot_parameters_histogram(
    parameters = [cosmo_params], 
    names = cosmo_params.dtype.names[:8], 
    bins = 50, 
    alpha = 0.7, 
    density = True
)
fig.tight_layout()

#### Plot all c000 HOD parameters vs HOD parameters returning nbar=0.0085 for c000

Imposing a physical constraint on the sampled HOD space through a density cut.

**First measuring the density for all sampled HODs for c000.**

In [ ]:
density_dir = Path(f'/pscratch/sd/s/sbouchar/test/bgs_nofilter/base/c000_ph000/seed0/') # Temporary, to test density filtering
density_fns = sorted(density_dir.glob('hod*/density.npy'))
densities = [np.load(fn) for fn in density_fns]
density_idx = [int(fn.stem.lstrip('hod')) for i, fn in enumerate(sorted(density_dir.glob('hod*'))) if densities[i] >= target]

keys = [f'c{c:03d}' for c in sorted(cosmo_list)]
hod_params_c000 = load_hod_params(parameters_dir / 'hod_params', keys=keys)['c000']
hod_params_density = hod_params_c000[density_idx]

fig, ax = plot_parameters_histogram(
    parameters = [hod_params_c000, hod_params_density], 
    names = hod_params_c000.dtype.names, 
    labels = ['c000', rf'c000 w/ $\bar{{n}} \geq {target:.2e}$'],
    colors = ['blue', 'red'],
    bins = 50, 
    alpha = 0.5, 
    # density = True,
)
ax[0].legend()
fig.tight_layout()

In [ ]:
# triangle version
params = ['logM_cut', 'logM_1', 'sigma', 'alpha', 'kappa']

fig, ax = plot_parameters_triangle(
    parameters = [hod_params_c000, hod_params_density], 
    names = params, 
    labels = ['c000', rf'c000 w/ $\bar{{n}} \geq {target:.2e}$'],
    colors = ['blue', 'red'],
    bins = 50, 
    alpha = 0.5, 
    s = 5,
)
fig.tight_layout()

**Then, the HOD space for the first 100 HOD reaching the target density of 0.0085 (Mpc/h)^-3 is selected for all cosmologies.** 

In [ ]:
keys = [f'c{c:03d}' for c in sorted(cosmo_list)]
hod_params_dict = load_hod_params(parameters_dir / 'hod_params', keys=keys)
hod_params_density = []
hod_params_bgs = []

for c in cosmo_list:
    hod_fns = sorted(measurements_dir.glob(f'base/c{c:03d}_ph000/seed0/hod*'))
    density_idx = [int(fn.stem.lstrip('hod')) for fn in hod_fns]
    
    if len(density_idx) == 0:
        continue

    hod_params_c = hod_params_dict[f'c{c:03d}'][density_idx]
    hod_params_density.append(hod_params_c)
    hod_params_bgs.append(hod_params_dict[f'c{c:03d}'])

hod_params_density = np.concatenate(hod_params_density) # HOD parameters across cosmologies with density cut
hod_params_bgs = np.concatenate(hod_params_bgs) # HOD parameters across all cosmologies

fig, ax = plot_parameters_histogram(
    parameters = [hod_params_bgs, hod_params_density], 
    names = hod_params_bgs.dtype.names, 
    labels = ['BGS', rf'BGS w/ $\bar{{n}} \geq {target:.2e}$'],
    colors = ['blue', 'red'],
    bins = 50, 
    alpha = 0.5, 
    # density = True,
)
ax[0].legend()
fig.tight_layout()

In [ ]:
# triangle version
params = ['logM_cut', 'logM_1', 'sigma', 'alpha', 'kappa']

fig, ax = plot_parameters_triangle(
    parameters = [hod_params_bgs, hod_params_density], 
    names = params, 
    labels = ['BGS', rf'BGS w/ $\bar{{n}} \geq {target:.2e}$'],
    colors = ['blue', 'red'],
    bins = 50, 
    alpha = 0.5, 
    s = 5,
)
fig.tight_layout()

## 2- Processed mocks

Count the number of HOD folders in each cosmology folder.

Then, count the number of statistic files computed (marginalized over HOD) for each cosmology.

#### Base boxes hod counts

In [ ]:
hod_fns = get_hod_folders(dir=measurements_dir, cosmologies=cosmo_list, phases=[0], seeds=[0], sim_type='base')

expected_number = 100
warn_empty = True
print_full = False

hod_count = {}
for cosmology in cosmo_list:
    count = len(hod_fns[cosmology][0][0])
    if count == 0 and warn_empty:
        print(f'Cosmology c{cosmology:03d} has no HOD folders.')
        continue
    last_hod = hod_fns[cosmology][0][0][-1].stem.lstrip('hod')
    if count == expected_number and print_full:
        print(f'Cosmology c{cosmology:03d} has {count} HOD folders. Last HOD: {last_hod}')
    elif count < expected_number and count > 0:
        print(f'Cosmology c{cosmology:03d} has {count} HOD folders (expected {expected_number}). Last HOD: {last_hod}')
    hod_count[cosmology] = count

if len(hod_count) == len(cosmo_list) and all([c == expected_number for c in hod_count.values()]):
    print(f'All cosmologies processed with {expected_number} HOD folders.')

In [ ]:
statistics = [
    'density',
    'tpcf_los_x',
    'tpcf_los_y',
    'tpcf_los_z',
    'quantile_data_correlation_los_x',
    'quantile_data_correlation_los_y',
    'quantile_data_correlation_los_z',
    'quantile_correlation_los_x',
    'quantile_correlation_los_y',
    'quantile_correlation_los_z',
    'power_spectrum_los_x',
    'power_spectrum_los_y',
    'power_spectrum_los_z',
    'quantile_data_power_los_x',
    'quantile_data_power_los_y',
    'quantile_data_power_los_z',
    'quantile_power_los_x',
    'quantile_power_los_y',
    'quantile_power_los_z'
]
expected_number = 100
warn_empty = True
print_full = False

stat_count = {statistic: {} for statistic in statistics}

for cosmology in cosmo_list:
    for statistic in statistics:
        count = 0
        for fn in hod_fns[cosmology][0][0]:
            stat_fns = sorted(fn.glob(f'{statistic}*.npy')) + sorted(fn.glob(f'{statistic}*.h5')) # npy or h5
            count += len(stat_fns)
        
        stat_count[statistic][cosmology] = count
        if count == expected_number and print_full:
            print(f'Cosmology c{cosmology:03d} has {count} {statistic} files.')
        elif count < expected_number and count > 0:
            print(f'Cosmology c{cosmology:03d} has {count} {statistic} files (expected {expected_number}).')
        elif count == 0 and warn_empty:
            print(f'Cosmology c{cosmology:03d} has no {statistic} files.')

# for statistic in statistics:
#     if len(stat_count[statistic]) == len(cosmo_list) and all([c == expected_number for c in stat_count[statistic].values()]):
#         print(f'All cosmologies processed with {expected_number} {statistic} files.')

## 3- Base boxes measurements

### 3.1- Density

In [ ]:
densities = get_densities(dir=measurements_dir, cosmologies=cosmo_list, phases=[0], seeds=[0], sim_type='base')

bins = np.linspace(0, 0.03, 100)
plt.hist(densities, bins=bins, label='Density distribution')
plt.axvline(2e-2, color='k', linestyle='--', alpha=0.8, label='BGS bright target density')
plt.axvline(1e-2, color='k', linestyle='-.', alpha=0.8, label='BGS faint target density')
plt.axvline(5e-3, color='k', linestyle='dotted', alpha=0.8, label='LRG target density')

plt.axvline(target, color='r', linestyle='solid', alpha=0.5, label=rf'$\bar{{n}} \geq {target:.1e}\ h^3 Mpc^{{-3}}$ cut-off')

plt.xlim(0, 0.03)
plt.legend()
plt.xlabel(r'Density ($h^3 Mpc^{-3}$)')
plt.ylabel('Count')
plt.title('Density distribution amongst simulated BGS base boxes')
plt.ticklabel_format(style='sci', axis='x', scilimits=(0,0));

### 3.2- Two-point correlation function

In [ ]:
cfs = get_tpcf(dir=measurements_dir, cosmologies=cosmo_list, phases=[0], seeds=[0], sim_type='base')

fig, ax = plt.subplots(figsize=(8,6))

plot_tpcf(ax, cfs)

if ref_dir is not None:
    fns = sorted(ref_dir.glob('tpcf_los_*.npy'))
    ref_cf = sum([TwoPointEstimator.load(fn).normalize() for fn in fns])
    plot_tpcf(ax, [ref_cf], c='k', ls='--', alpha=1, add_handles=False)

ax.set_xlabel(r's ($h^{-1}$ Mpc)')
ax.set_ylabel(r's$^2 \xi_{\ell}(s)$ ($h^{-1}$ Mpc)$^2$')
ax.set_title('Two-point correlation functions from ACM BGS base boxes');

# plt.ylim(-100, 100);
# plt.xlim(0, 30);

### 3.3- Density split

In [ ]:
quantiles = [0, 1, 2, 3, 4]
cc_cfs = get_ds_cf(stat_name='quantile_data_correlation', dir=measurements_dir, cosmologies=cosmo_list, phases=[0], seeds=[0], sim_type='base', quantiles=quantiles)
ac_cfs = get_ds_cf(stat_name='quantile_correlation', dir=measurements_dir, cosmologies=cosmo_list, phases=[0], seeds=[0], sim_type='base', quantiles=quantiles)

fig, ax = plt.subplots(1, 2, figsize=(12,5), sharex=True)

plot_ds_cf(ax[0], cc_cfs, add_handles=False)
plot_ds_cf(ax[1], ac_cfs)

if ref_dir is not None:
    fn_list_cc = sorted(ref_dir.glob('quantile_data_correlation_los_*.npy'))
    fn_list_ac = sorted(ref_dir.glob('quantile_correlation_los_*.npy'))
    ref_ccf = [sum([np.load(fn, allow_pickle=True)[q].normalize() for fn in fn_list_cc]) for q in quantiles]
    ref_acf = [sum([np.load(fn, allow_pickle=True)[q].normalize() for fn in fn_list_ac]) for q in quantiles]

    plot_ds_cf(ax[0], [ref_ccf], c='k', ls='--', alpha=1, add_handles=False)
    plot_ds_cf(ax[1], [ref_acf], c='k', ls='--', alpha=1, add_handles=False)

for a in ax:
    a.set_xlabel(r's ($h^{-1}$ Mpc)')
    a.set_ylabel(r's$^2 \xi_{0}(s)$ ($h^{-1}$ Mpc)$^2$')

ax[0].set_title('Cross-correlation')
ax[1].set_title('Auto-correlation')

fig.suptitle('DensitySplit Correlation Functions from ACM BGS base boxes', fontsize=14)
fig.tight_layout()

### 3.4- Power spectrum

In [ ]:
# pks = get_power_spectra(dir=measurements_dir, cosmologies=cosmo_list, phases=[0], seeds=[0], sim_type='base')

fig, ax = plt.subplots(figsize=(8,6))

plot_power_spectra(ax, pks, ells=(0,2))

if ref_dir is not None:
    fns = sorted(ref_dir.glob('power_spectrum_los_*.h5'))
    ref_pk = sum([lsstypes.read(fn) for fn in fns])
    plot_power_spectra(ax, [ref_pk], c='k', ls='--', alpha=1, add_handles=False)

# ax.set_xscale('log')
ax.set_xlabel(r'k ($h^{-1}$ Mpc)')
ax.set_ylabel(r'k$^2 \P_{\ell}(k)$ ($h^{-1}$ Mpc)$^3$')
ax.set_title('Power Spectra functions from ACM BGS base boxes');

# ax.set_xlim(0.01, 0.7);

### 3.5- Density split power spectrum

In [ ]:
quantiles = [0, 1, 2, 3, 4]
pks_qq = get_ds_power_spectra(stat_name='quantile_power', dir=measurements_dir, cosmologies=[0], phases=[0], seeds=[0], sim_type='base', quantiles=quantiles)
pks_qd = get_ds_power_spectra(stat_name='quantile_data_power', dir=measurements_dir, cosmologies=[0], phases=[0], seeds=[0], sim_type='base', quantiles=quantiles)

fig, ax = plt.subplots(1, 2, figsize=(12,5), sharex=True)

plot_ds_power_spectra(ax[0], pks_qd, add_handles=False)
plot_ds_power_spectra(ax[1], pks_qq)

if ref_dir is not None:
    fn_list_qd = sorted(ref_dir.glob('quantile_data_power_los_*.h5'))
    fn_list_qq = sorted(ref_dir.glob('quantile_power_los_*.h5'))
    ref_pk_qd = [sum([lsstypes.read(fn).get(quantiles=q) for fn in fn_list_qd]) for q in quantiles]
    ref_pk_qq = [sum([lsstypes.read(fn).get(quantiles=q) for fn in fn_list_qq]) for q in quantiles]

    plot_ds_power_spectra(ax[0], [ref_pk_qd], c='k', ls='--', alpha=1, add_handles=False)
    plot_ds_power_spectra(ax[1], [ref_pk_qq], c='k', ls='--', alpha=1, add_handles=False)

for a in ax:
    a.set_xscale('log')
    a.set_xlabel(r'k ($h^{-1}$ Mpc)')
    a.set_ylabel(r'k$^2 \P_{\ell}(k)$ ($h^{-1}$ Mpc)$^3$')

ax[0].set_title('Cross-correlation')
ax[1].set_title('Auto-correlation')

fig.suptitle('DensitySplit Power Spectra from ACM BGS base boxes', fontsize=14)
fig.tight_layout()

### Interactive visualization of the measurements

In [ ]:
plot_interactive(
    stat_name = 'tpcf', # Change this as needed
    dir = measurements_dir, 
    cosmologies = cosmo_list, 
    phases = [0], 
    seeds = [0], 
    sim_type = 'base', 
    alpha = 1,
)

## 4- Best-fit HOD comparison
*See `--plot` option from `best_fit.py` to get this figure.*

<img src="jobs/bgs-20/best_fit_c000_ells(0, 2).png" alt="Best-fit HOD comparison figure" width="600"/>

## 5- Small boxes measurements

In [ ]:
bestfit_hod = 70
abacus_dir = Path('/global/cfs/cdirs/desi/cosmosim/Abacus/small/')

phase_list = get_abacus_phases(abacus_dir, z=0.2, cosmo=0)[1] 
print(f'Found {len(phase_list)} phases for cosmo 0 at z=0.2')

In [ ]:
hod_fns = get_hod_folders(dir=measurements_dir, cosmologies=[0], phases=phase_list, seeds=[0], sim_type='small', hod_idx=bestfit_hod)

phases = []
expected_stat_number = 1 + 3*(2+1) # density + 3 los * (tpcf + 2 quantile correlations) 
for phase in phase_list:
    fn_ = hod_fns[0][phase][0]
    if len(fn_) != 0:
        fn = fn_[0]
        file_count = len(sorted(fn.glob('*')))
        if file_count == expected_stat_number:
            phases.append(phase)
        elif file_count != 0:
            print(f'Phase {phase} has incomplete statistics: {file_count} files (expected {expected_stat_number}).')
            
missing = set(phase_list) - set(phases)
print(f'Found {len(phases)} phases with HOD realizations out of {len(phase_list)} total phases.')
if len(missing) > 0:
    print(f'Missing {len(missing)} phases: {missing}')

### 5.1- Density

In [ ]:
densities = get_densities(dir=measurements_dir, cosmologies=[0], phases=phase_list, seeds=[0], sim_type='small', hod_idx=bestfit_hod)

bins = np.linspace(0, 0.03, 100)
plt.hist(densities, bins=bins, label='Density distribution')
plt.axvline(2e-2, color='k', linestyle='--', alpha=0.8, label='BGS bright target density')
plt.axvline(1e-2, color='k', linestyle='-.', alpha=0.8, label='BGS faint target density')
plt.axvline(5e-3, color='k', linestyle='dotted', alpha=0.8, label='LRG target density')

plt.axvline(target, color='r', linestyle='solid', alpha=0.8, label=rf'$\bar{{n}} \geq {target:.1e}\ h^3 Mpc^{{-3}}$ cut-off')

plt.xlim(0, 0.03)
plt.legend()
plt.xlabel(r'Density ($h^3 Mpc^{-3}$)')
plt.ylabel('Count')
plt.title('Density distribution amongst simulated BGS small boxes')
plt.ticklabel_format(style='sci', axis='x', scilimits=(0,0));

### 5.2- Two-point correlation function

In [ ]:
cfs = get_tpcf(dir=measurements_dir, cosmologies=[0], phases=phase_list, seeds=[0], sim_type='small', hod_idx=bestfit_hod)

fig, ax = plt.subplots(figsize=(8,6))

plot_tpcf(ax, cfs)

if ref_dir is not None:
    fns = sorted(ref_dir.glob('tpcf_los_*.npy'))
    ref_cf = sum([TwoPointEstimator.load(fn).normalize() for fn in fns])
    plot_tpcf(ax, [ref_cf], c='k', ls='--', alpha=1, add_handles=False)
    
plt.xlabel(r's ($h^{-1}$ Mpc)')
plt.ylabel(r's$^2 \xi_{\ell}(s)$ ($h^{-1}$ Mpc)$^2$')
plt.title(f'Two-point correlation functions from ACM BGS small boxes (hod{bestfit_hod})');

# plt.ylim(-100, 100);
# plt.xlim(0, 30);

### 5.3- Density split

In [ ]:
quantiles = [0, 1, 2, 3, 4] 
cc_cfs = get_ds_cf(stat_name='quantile_data_correlation', dir=measurements_dir, cosmologies=[0], phases=phase_list, seeds=[0], sim_type='small', quantiles=quantiles, hod_idx=bestfit_hod)
ac_cfs = get_ds_cf(stat_name='quantile_correlation', dir=measurements_dir, cosmologies=[0], phases=phase_list, seeds=[0], sim_type='small', quantiles=quantiles, hod_idx=bestfit_hod)

fig, ax = plt.subplots(1, 2, figsize=(12,5), sharex=True)

plot_ds_cf(ax[0], cc_cfs, add_handles=False)
plot_ds_cf(ax[1], ac_cfs)

if ref_dir is not None:
    fn_list_cc = sorted(ref_dir.glob('quantile_data_correlation_los_*.npy'))
    fn_list_ac = sorted(ref_dir.glob('quantile_correlation_los_*.npy'))
    ref_ccf = [sum([np.load(fn, allow_pickle=True)[q].normalize() for fn in fn_list_cc]) for q in quantiles]
    ref_acf = [sum([np.load(fn, allow_pickle=True)[q].normalize() for fn in fn_list_ac]) for q in quantiles]

    plot_ds_cf(ax[0], [ref_ccf], c='k', ls='--', alpha=1, add_handles=False)
    plot_ds_cf(ax[1], [ref_acf], c='k', ls='--', alpha=1, add_handles=False)

for a in ax:
    a.set_xlabel(r's ($h^{-1}$ Mpc)')
    a.set_ylabel(r's$^2 \xi_{0}(s)$ ($h^{-1}$ Mpc)$^2$')

ax[0].set_title('Cross-correlation')
ax[1].set_title('Auto-correlation')

fig.suptitle('DensitySplit Correlation Functions from ACM BGS small boxes', fontsize=14)
fig.tight_layout()

### 5.4- Power spectrum

In [ ]:
pks = get_power_spectra(dir=measurements_dir, cosmologies=[0], phases=phase_list, seeds=[0], sim_type='small', hod_idx=bestfit_hod)

fig, ax = plt.subplots(figsize=(8,6))

plot_power_spectra(ax, pks, ells=(0,2))

if ref_dir is not None:
    fns = sorted(ref_dir.glob('power_spectrum_los_*.h5'))
    ref_cf = sum([lsstypes.read(fn) for fn in fns])
    plot_power_spectra(ax, [ref_cf], c='k', ls='--', alpha=1, add_handles=False)

# ax.set_xscale('log')
ax.set_xlabel(r'k ($h^{-1}$ Mpc)')
ax.set_ylabel(r'k$^2 \P_{\ell}(k)$ ($h^{-1}$ Mpc)$^3$')
ax.set_title('Power Spectra functions from ACM BGS small boxes');

In [ ]:
quantiles = [0, 1, 2, 3, 4]
pks_qq = get_ds_power_spectra(stat_name='quantile_power', dir=measurements_dir, cosmologies=[0], phases=phase_list, seeds=[0], sim_type='small', quantiles=quantiles)
pks_qd = get_ds_power_spectra(stat_name='quantile_data_power', dir=measurements_dir, cosmologies=[0], phases=phase_list, seeds=[0], sim_type='small', quantiles=quantiles)

fig, ax = plt.subplots(1, 2, figsize=(12,5), sharex=True)

plot_ds_power_spectra(ax[0], pks_qd, add_handles=False)
plot_ds_power_spectra(ax[1], pks_qq)

if ref_dir is not None:
    fn_list_qd = sorted(ref_dir.glob('quantile_data_power_los_*.h5'))
    fn_list_qq = sorted(ref_dir.glob('quantile_power_los_*.h5'))
    ref_pk_qd = [sum([lsstypes.read(fn).get(quantiles=q) for fn in fn_list_qd]) for q in quantiles]
    ref_pk_qq = [sum([lsstypes.read(fn).get(quantiles=q) for fn in fn_list_qq]) for q in quantiles]

    plot_ds_power_spectra(ax[0], [ref_pk_qd], c='k', ls='--', alpha=1, add_handles=False)
    plot_ds_power_spectra(ax[1], [ref_pk_qq], c='k', ls='--', alpha=1, add_handles=False)

for a in ax:
    a.set_xscale('log')
    a.set_xlabel(r'k ($h^{-1}$ Mpc)')
    a.set_ylabel(r'k$^2 \P_{\ell}(k)$ ($h^{-1}$ Mpc)$^3$')

ax[0].set_title('Cross-correlation')
ax[1].set_title('Auto-correlation')

fig.suptitle('DensitySplit Power Spectra from ACM BGS small boxes', fontsize=14)
fig.tight_layout()

### Interractive vizualisations

In [ ]:
plot_interactive(
    stat_name = 'tpcf', 
    dir = measurements_dir, 
    cosmologies = [0], 
    phases = phase_list, 
    seeds = [0], 
    sim_type = 'small', 
    hod_idx = bestfit_hod, 
    alpha = 1,
)

In [ ]:
plot_interactive(
    stat_name = 'quantile_correlation', 
    dir = measurements_dir, 
    cosmologies = [0], 
    phases = phase_list, 
    seeds = [0], 
    sim_type = 'small', 
    hod_idx = bestfit_hod, 
    alpha = 1,
)

In [ ]:
plot_interactive(
    stat_name = 'quantile_data_correlation', 
    dir = measurements_dir, 
    cosmologies = [0], 
    phases = phase_list, 
    seeds = [0], 
    sim_type = 'small', 
    hod_idx = bestfit_hod, 
    alpha = 1,
)

In [ ]:
plot_interactive(
    stat_name = 'power_spectrum', 
    dir = measurements_dir, 
    cosmologies = [0], 
    phases = phase_list, 
    seeds = [0], 
    sim_type = 'small', 
    hod_idx = bestfit_hod, 
    alpha = 1,
)

## 6- Compressed files

In [ ]:
from acm.observables.bgs import tpcf, ds_xiqq, ds_xiqg

compressed_dir = Path(f'/pscratch/sd/s/sbouchar/acm/bgs{Mr}/input_data/')
paths = dict(data_dir=compressed_dir)

In [ ]:
obs = tpcf(flat_output_dims=None, paths=paths)
s = obs.s.values
cosmos = obs.cosmo_idx.values
hods = obs.hod_idx.values
ells = obs.ells.values

for cosmo in cosmos:
    for hod in hods:
        for i, ell in enumerate(ells):
            pole = obs.y.sel(cosmo_idx=cosmo, hod_idx=hod, ells=ell)
            plt.plot(s, pole*s**2, c=f'C{i}', alpha=0.1)

handles = [
    plt.Line2D([0], [0], color=f'C{i}', lw=2, label=rf'$\ell={{{i*2}}}$') for i in range(3)
]
plt.legend(handles=handles)

plt.xlabel(r's ($h^{-1}$ Mpc)')
plt.ylabel(r's$^2 \xi_{\ell}(s)$ ($h^{-1}$ Mpc)$^2$')
plt.title('Two-point correlation functions from ACM BGS base boxes (from observables module)');

In [ ]:
obs_cc = ds_xiqq(flat_output_dims=None, paths=paths)
obs_ac = ds_xiqg(flat_output_dims=None, paths=paths)
s = obs_cc.s.values
cosmos = obs_cc.cosmo_idx.values
hods = obs_cc.hod_idx.values
quantiles = obs_cc.quantiles.values
ells = obs_cc.ells.values

fig, ax = plt.subplots(1, 2, figsize=(12,5), sharex=True)
for cosmo in cosmos:
    for hod in hods:
        for i, q in enumerate(quantiles):
            pole = obs_cc.y.sel(cosmo_idx=cosmo, hod_idx=hod, quantiles=q, ells=0)
            ax[0].plot(s, pole*s**2, c=f'C{i}', alpha=0.1)
            
            pole = obs_ac.y.sel(cosmo_idx=cosmo, hod_idx=hod, quantiles=q, ells=0)
            ax[1].plot(s, pole*s**2, c=f'C{i}', alpha=0.1)
handles = [
    plt.Line2D([0], [0], color=f'C{i}', lw=2, label=f'DS{q}') for i, q in enumerate(quantiles)
]
ax[1].legend(handles=handles, title='Quantiles', bbox_to_anchor=(1.05, 1), loc='upper left')
ax[0].set_title('Cross-correlation')
ax[1].set_title('Auto-correlation')
for a in ax:
    a.set_xlabel(r's ($h^{-1}$ Mpc)')
    a.set_ylabel(r's$^2 \xi_{0}(s)$ ($h^{-1}$ Mpc)$^2$')
fig.suptitle('DensitySplit Correlation Functions from ACM BGS base boxes (from observables module)', fontsize=14);

### Temporary: Fourier-space corrupted files

In [ ]:
import lsstypes
import numpy as np
from visual_tools import nested_set

Mr = -21.35
data_dir = Path(f'/pscratch/sd/s/sbouchar/acm/bgs{Mr}/measurements/')
simtype = 'base' # small
c = cosmo_list # [0]
p = [0] # phase_list
s = [0]

stat_name = '*' # Check all statistics for outliers

outliers_dict = {}
for cosmo in c:
    for phase in p:
        for seed in s:
            hod_fns = sorted((data_dir / f'{simtype}/c{cosmo:03d}_ph{phase:03d}/seed{seed}/').glob('hod*'))
            if len(hod_fns) == 0:
                print(f'No HOD folders found for cosmology {cosmo} and phase {phase}.')
                continue
            
            hods = []
            for hod_fn in hod_fns:
                fns = sorted(hod_fn.glob(f'{stat_name}.h5'))
                for fn in fns:
                    try:
                        lsstypes.read(fn)
                    except OSError:
                        hods.append(int(hod_fn.stem.lstrip('hod')))
            
            if len(hods) > 0:
                print(f'Found {len(hods)} outlier HODs for cosmology {cosmo} and phase {phase}: {hods}')
                nested_set(outliers_dict, [str(cosmo), str(phase), str(seed)], hods, extend=True)

if len(outliers_dict) == 0:
    print('No outliers found across checked cosmologies, phases, and seeds.')
else:
    np.save(f'jobs/bgs{Mr}/outliers/all_corrupted_fourier_{simtype}.npy', outliers_dict)